In [0]:
# ============================================================
# BANKING GENAI USE CASE
# DATABRICKS SILVER -> GOLD
#
# Silver:
#   ujjivan_2.silver.bank_transaction_fraud_detection
#
# Gold:
#   S3 Location: s3://ujjivanpoc/Gold
#   Databricks Schema: ujjivan_2.gold
#
# Gold Tables:
#   1. customer360
#   2. spending_summary
#   3. merchant_affinity
#   4. digital_engagement
#   5. customer_segmentation
#   6. customer_risk
#   7. product_affinity
#   8. next_best_action
#   9. campaign_target_list
#
# Purpose:
#   Customer 360
#   Cross Sell
#   Up Sell
#   Next Best Action
#   Personalized Engagement
# ============================================================


from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

SILVER_TABLE = "ujjivan_2.silver.bank_transaction_fraud_detection"

GOLD_SCHEMA = "ujjivan_2.gold"

# Main S3 Gold location
GOLD_S3_PATH = "s3://ujjivanpoc/Gold"


# ============================================================
# 2. CREATE GOLD SCHEMA
# ============================================================

print("")
print("=" * 70)
print("CREATING GOLD SCHEMA")
print("=" * 70)

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
""")

print("Gold schema ready:", GOLD_SCHEMA)
print("Gold S3 location:", GOLD_S3_PATH)


# ============================================================
# 3. READ SILVER TABLE
# ============================================================

print("")
print("=" * 70)
print("READING SILVER TABLE")
print("=" * 70)

print("Silver table:", SILVER_TABLE)

silver_df = spark.table(SILVER_TABLE)

silver_count = silver_df.count()

print("Silver record count:", silver_count)

display(
    silver_df.limit(10)
)


# ============================================================
# 4. STANDARDIZE DATA TYPES
# ============================================================

print("")
print("=" * 70)
print("STANDARDIZING DATA TYPES")
print("=" * 70)

df = (
    silver_df

    .withColumn(
        "Transaction_Date",
        F.to_date(
            F.col("Transaction_Date")
        )
    )

    .withColumn(
        "Transaction_Amount",
        F.col("Transaction_Amount")
        .cast("double")
    )

    .withColumn(
        "Account_Balance",
        F.col("Account_Balance")
        .cast("double")
    )

    .withColumn(
        "Age",
        F.col("Age")
        .cast("int")
    )

    .withColumn(
        "Is_Fraud",
        F.col("Is_Fraud")
        .cast("int")
    )
)


# ============================================================
# 5. REMOVE DUPLICATES
# ============================================================

print("")
print("Removing duplicate transactions...")

df = (
    df
    .dropDuplicates(
        ["Transaction_ID"]
    )
)


# ============================================================
# 6. REMOVE INVALID RECORDS
# ============================================================

print("Removing invalid records...")

df = (
    df

    .filter(
        F.col("Customer_ID").isNotNull()
    )

    .filter(
        F.col("Transaction_ID").isNotNull()
    )

    .filter(
        F.col("Transaction_Amount").isNotNull()
    )
)


# ============================================================
# 7. STANDARDIZE STRING COLUMNS
# ============================================================

print("Standardizing string columns...")

df = (
    df

    .withColumn(
        "Transaction_Type",
        F.upper(
            F.trim(
                F.col("Transaction_Type")
            )
        )
    )

    .withColumn(
        "Merchant_Category",
        F.upper(
            F.trim(
                F.col("Merchant_Category")
            )
        )
    )

    .withColumn(
        "Account_Type",
        F.upper(
            F.trim(
                F.col("Account_Type")
            )
        )
    )

    .withColumn(
        "Gender",
        F.upper(
            F.trim(
                F.col("Gender")
            )
        )
    )
)


# ============================================================
# 8. NORMALIZE TRANSACTION TYPES
# ============================================================

print("Normalizing transaction types...")

df = (
    df

    .withColumn(
        "Transaction_Type_Normalized",

        F.when(
            F.col("Transaction_Type").isin(
                "CREDIT",
                "CR",
                "DEPOSIT"
            ),
            "CREDIT"
        )

        .when(
            F.col("Transaction_Type").isin(
                "DEBIT",
                "DB",
                "WITHDRAWAL"
            ),
            "DEBIT"
        )

        .when(
            F.col("Transaction_Type") == "TRANSFER",
            "TRANSFER"
        )

        .when(
            F.col("Transaction_Type") == "BILL PAYMENT",
            "BILL_PAYMENT"
        )

        .otherwise(
            F.col("Transaction_Type")
        )
    )
)


# ============================================================
# 9. TRANSACTION FEATURES
# ============================================================

print("Creating transaction features...")

df = (
    df

    # --------------------------------------------------------
    # Transaction Month
    # --------------------------------------------------------

    .withColumn(
        "Transaction_Month",
        F.date_format(
            F.col("Transaction_Date"),
            "yyyy-MM"
        )
    )

    # --------------------------------------------------------
    # Weekend Flag
    # --------------------------------------------------------

    .withColumn(
        "Weekend_Flag",

        F.when(
            F.dayofweek(
                F.col("Transaction_Date")
            ).isin(1, 7),
            "YES"
        )

        .otherwise(
            "NO"
        )
    )

    # --------------------------------------------------------
    # High Value Transaction
    # --------------------------------------------------------

    .withColumn(
        "High_Value_Flag",

        F.when(
            F.col("Transaction_Amount") >= 50000,
            "YES"
        )

        .otherwise(
            "NO"
        )
    )

    # --------------------------------------------------------
    # Fraud Status
    # --------------------------------------------------------

    .withColumn(
        "Fraud_Status",

        F.when(
            F.col("Is_Fraud") == 1,
            "FRAUD"
        )

        .otherwise(
            "NORMAL"
        )
    )
)


# ============================================================
# 10. DIGITAL CHANNEL FLAG
# ============================================================

print("Creating digital channel flag...")

digital_devices = [
    "Mobile Device",
    "POS Mobile App",
    "Voice Assistant",
    "Payment Gateway Device",
    "Virtual Card"
]


df = (
    df

    .withColumn(
        "Digital_Channel_Flag",

        F.when(
            F.col("Transaction_Device")
            .isin(digital_devices),

            "YES"
        )

        .otherwise(
            "NO"
        )
    )
)


# ============================================================
# 11. CUSTOMER 360
# ============================================================

print("")
print("=" * 70)
print("CREATING CUSTOMER 360")
print("=" * 70)


# ------------------------------------------------------------
# Latest transaction/customer record
# ------------------------------------------------------------

customer_window = (
    Window
    .partitionBy(
        "Customer_ID"
    )
    .orderBy(
        F.col(
            "Transaction_Date"
        ).desc(),

        F.col(
            "Transaction_Time"
        ).desc()
    )
)


latest_customer = (
    df

    .withColumn(
        "rn",

        F.row_number()
        .over(customer_window)
    )

    .filter(
        F.col("rn") == 1
    )

    .select(
        "Customer_ID",
        "Customer_Name",
        "Gender",
        "Age",
        "State",
        "City",
        "Bank_Branch",
        "Account_Type",
        "Account_Balance",
        "Customer_Email",
        "Customer_Contact"
    )
)


# ------------------------------------------------------------
# Customer Metrics
# ------------------------------------------------------------

customer_metrics = (
    df

    .groupBy(
        "Customer_ID"
    )

    .agg(

        F.countDistinct(
            "Transaction_ID"
        ).alias(
            "Total_Transactions"
        ),

        F.sum(
            "Transaction_Amount"
        ).alias(
            "Total_Spend"
        ),

        F.avg(
            "Transaction_Amount"
        ).alias(
            "Average_Transaction_Amount"
        ),

        F.max(
            "Transaction_Amount"
        ).alias(
            "Maximum_Transaction_Amount"
        ),

        F.min(
            "Transaction_Amount"
        ).alias(
            "Minimum_Transaction_Amount"
        ),

        F.max(
            "Transaction_Date"
        ).alias(
            "Last_Transaction_Date"
        ),

        F.sum(
            F.when(
                F.col("High_Value_Flag") == "YES",
                1
            )
            .otherwise(0)
        ).alias(
            "High_Value_Transaction_Count"
        ),

        F.sum(
            F.when(
                F.col("Is_Fraud") == 1,
                1
            )
            .otherwise(0)
        ).alias(
            "Fraud_Transaction_Count"
        ),

        F.sum(
            F.when(
                F.col("Digital_Channel_Flag") == "YES",
                1
            )
            .otherwise(0)
        ).alias(
            "Digital_Transaction_Count"
        ),

        F.countDistinct(
            "Merchant_Category"
        ).alias(
            "Merchant_Category_Count"
        )
    )
)


# ------------------------------------------------------------
# Create Customer 360
# ------------------------------------------------------------

customer360 = (
    latest_customer

    .join(
        customer_metrics,
        "Customer_ID",
        "left"
    )
)


# ============================================================
# 12. SPENDING SUMMARY
# ============================================================

print("")
print("=" * 70)
print("CREATING SPENDING SUMMARY")
print("=" * 70)


spending_summary = (
    df

    .groupBy(
        "Customer_ID"
    )

    .agg(

        F.sum(
            "Transaction_Amount"
        ).alias(
            "Total_Spend"
        ),

        F.avg(
            "Transaction_Amount"
        ).alias(
            "Average_Spend"
        ),

        F.max(
            "Transaction_Amount"
        ).alias(
            "Maximum_Spend"
        ),

        F.min(
            "Transaction_Amount"
        ).alias(
            "Minimum_Spend"
        ),

        F.sum(
            F.when(
                F.col("Weekend_Flag") == "YES",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Weekend_Spend"
        ),

        F.sum(
            F.when(
                F.col("High_Value_Flag") == "YES",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "High_Value_Spend"
        ),

        F.sum(
            F.when(
                F.col("Merchant_Category") == "RESTAURANT",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Restaurant_Spend"
        ),

        F.sum(
            F.when(
                F.col("Merchant_Category") == "GROCERIES",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Grocery_Spend"
        ),

        F.sum(
            F.when(
                F.col("Merchant_Category") == "ENTERTAINMENT",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Entertainment_Spend"
        ),

        F.sum(
            F.when(
                F.col("Merchant_Category") == "CLOTHING",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Clothing_Spend"
        ),

        F.sum(
            F.when(
                F.col("Merchant_Category") == "HEALTH",
                F.col("Transaction_Amount")
            )
            .otherwise(0)
        ).alias(
            "Health_Spend"
        )
    )
)


# ============================================================
# 13. MERCHANT AFFINITY
# ============================================================

print("")
print("=" * 70)
print("CREATING MERCHANT AFFINITY")
print("=" * 70)


merchant_frequency = (
    df

    .groupBy(
        "Customer_ID",
        "Merchant_Category"
    )

    .agg(

        F.count("*").alias(
            "Transaction_Count"
        ),

        F.sum(
            "Transaction_Amount"
        ).alias(
            "Category_Spend"
        )
    )
)


merchant_window = (
    Window

    .partitionBy(
        "Customer_ID"
    )

    .orderBy(

        F.col(
            "Transaction_Count"
        ).desc(),

        F.col(
            "Category_Spend"
        ).desc()
    )
)


merchant_affinity = (
    merchant_frequency

    .withColumn(
        "rn",

        F.row_number()
        .over(merchant_window)
    )

    .filter(
        F.col("rn") == 1
    )

    .select(

        "Customer_ID",

        F.col(
            "Merchant_Category"
        )
        .alias(
            "Preferred_Merchant_Category"
        ),

        F.col(
            "Transaction_Count"
        )
        .alias(
            "Preferred_Category_Transaction_Count"
        ),

        F.col(
            "Category_Spend"
        )
        .alias(
            "Preferred_Category_Spend"
        )
    )
)


# ============================================================
# 14. DIGITAL ENGAGEMENT
# ============================================================

print("")
print("=" * 70)
print("CREATING DIGITAL ENGAGEMENT")
print("=" * 70)


digital_engagement = (
    df

    .groupBy(
        "Customer_ID"
    )

    .agg(

        F.count("*").alias(
            "Total_Transactions"
        ),

        F.sum(
            F.when(
                F.col("Digital_Channel_Flag") == "YES",
                1
            )
            .otherwise(0)
        ).alias(
            "Digital_Transactions"
        ),

        F.countDistinct(
            "Transaction_Device"
        ).alias(
            "Device_Type_Count"
        )
    )

    .withColumn(
        "Digital_Engagement_Score",

        F.round(

            (
                F.col(
                    "Digital_Transactions"
                )

                /

                F.col(
                    "Total_Transactions"
                )
            )

            * 100,

            2
        )
    )

    .withColumn(
        "Digital_Customer_Flag",

        F.when(
            F.col(
                "Digital_Engagement_Score"
            ) >= 60,

            "YES"
        )

        .otherwise(
            "NO"
        )
    )
)


# ============================================================
# 15. PREFERRED DEVICE
# ============================================================

print("Creating preferred device...")

device_count = (
    df

    .groupBy(
        "Customer_ID",
        "Transaction_Device"
    )

    .agg(
        F.count("*")
        .alias(
            "Device_Usage"
        )
    )
)


device_window = (
    Window

    .partitionBy(
        "Customer_ID"
    )

    .orderBy(
        F.col(
            "Device_Usage"
        ).desc()
    )
)


preferred_device = (
    device_count

    .withColumn(
        "rn",

        F.row_number()
        .over(device_window)
    )

    .filter(
        F.col("rn") == 1
    )

    .select(

        "Customer_ID",

        F.col(
            "Transaction_Device"
        )
        .alias(
            "Preferred_Device"
        )
    )
)


digital_engagement = (
    digital_engagement

    .join(
        preferred_device,
        "Customer_ID",
        "left"
    )
)


# ============================================================
# 16. CUSTOMER SEGMENTATION
# ============================================================

print("")
print("=" * 70)
print("CREATING CUSTOMER SEGMENTATION")
print("=" * 70)


customer_segmentation = (
    customer360

    .withColumn(
        "Age_Group",

        F.when(
            F.col("Age") < 25,
            "YOUNG"
        )

        .when(
            F.col("Age") < 40,
            "ADULT"
        )

        .when(
            F.col("Age") < 60,
            "MIDDLE_AGE"
        )

        .otherwise(
            "SENIOR"
        )
    )

    .withColumn(
        "Balance_Segment",

        F.when(
            F.col("Account_Balance") >= 1000000,
            "HIGH_NET_WORTH"
        )

        .when(
            F.col("Account_Balance") >= 500000,
            "PREMIUM"
        )

        .when(
            F.col("Account_Balance") >= 100000,
            "MASS_AFFLUENT"
        )

        .otherwise(
            "STANDARD"
        )
    )

    .withColumn(
        "Customer_Value_Segment",

        F.when(
            F.col("Total_Spend") >= 1000000,
            "HIGH_VALUE"
        )

        .when(
            F.col("Total_Spend") >= 500000,
            "MEDIUM_VALUE"
        )

        .otherwise(
            "STANDARD_VALUE"
        )
    )
)


# ============================================================
# 17. CUSTOMER RISK
# ============================================================

print("")
print("=" * 70)
print("CREATING CUSTOMER RISK")
print("=" * 70)


customer_risk = (
    customer360

    .withColumn(
        "Risk_Score",

        F.when(
            F.col(
                "Fraud_Transaction_Count"
            ) >= 3,

            90
        )

        .when(
            F.col(
                "Fraud_Transaction_Count"
            ) == 2,

            70
        )

        .when(
            F.col(
                "Fraud_Transaction_Count"
            ) == 1,

            50
        )

        .otherwise(
            10
        )
    )

    .withColumn(
        "Risk_Level",

        F.when(
            F.col("Risk_Score") >= 70,
            "HIGH"
        )

        .when(
            F.col("Risk_Score") >= 40,
            "MEDIUM"
        )

        .otherwise(
            "LOW"
        )
    )
)


# ============================================================
# 18. PRODUCT AFFINITY
# ============================================================

print("")
print("=" * 70)
print("CREATING PRODUCT AFFINITY")
print("=" * 70)


# ------------------------------------------------------------
# Do NOT bring Total_Spend from spending_summary.
# customer360 already contains Total_Spend.
# ------------------------------------------------------------

spending_for_product = (
    spending_summary

    .select(

        "Customer_ID",

        "Restaurant_Spend",
        "Grocery_Spend",
        "Entertainment_Spend",
        "Clothing_Spend",
        "Health_Spend",

        "Weekend_Spend",
        "High_Value_Spend",

        "Average_Spend",
        "Maximum_Spend"
    )
)


# ------------------------------------------------------------
# Segmentation columns only
# ------------------------------------------------------------

segmentation_for_product = (
    customer_segmentation

    .select(

        "Customer_ID",

        "Age_Group",
        "Balance_Segment",
        "Customer_Value_Segment"
    )
)


# ------------------------------------------------------------
# Digital columns only
# ------------------------------------------------------------

digital_for_product = (
    digital_engagement

    .select(

        "Customer_ID",

        "Digital_Engagement_Score",
        "Digital_Customer_Flag",
        "Preferred_Device"
    )
)


# ------------------------------------------------------------
# Build Product Affinity
# ------------------------------------------------------------

product_affinity = (
    customer360

    .join(
        spending_for_product,
        "Customer_ID",
        "left"
    )

    .join(
        merchant_affinity,
        "Customer_ID",
        "left"
    )

    .join(
        segmentation_for_product,
        "Customer_ID",
        "left"
    )

    .join(
        digital_for_product,
        "Customer_ID",
        "left"
    )
)


# ============================================================
# 19. PRODUCT RECOMMENDATION
# ============================================================

print("Creating product recommendations...")


product_affinity = (
    product_affinity

    .withColumn(

        "Recommended_Product",

        # ----------------------------------------------------
        # Restaurant customer
        # ----------------------------------------------------

        F.when(

            (
                F.col(
                    "Restaurant_Spend"
                )

                >

                F.col(
                    "Total_Spend"
                ) * 0.30
            ),

            "Premium Dining Credit Card"
        )

        # ----------------------------------------------------
        # Healthcare customer
        # ----------------------------------------------------

        .when(

            (
                F.col(
                    "Health_Spend"
                )

                >

                F.col(
                    "Total_Spend"
                ) * 0.20
            ),

            "Health Insurance"
        )

        # ----------------------------------------------------
        # High balance customer
        # ----------------------------------------------------

        .when(

            F.col(
                "Account_Balance"
            ) >= 500000,

            "Wealth Management"
        )

        # ----------------------------------------------------
        # Business account
        # ----------------------------------------------------

        .when(

            F.col(
                "Account_Type"
            ) == "BUSINESS",

            "Business Loan"
        )

        # ----------------------------------------------------
        # Entertainment customer
        # ----------------------------------------------------

        .when(

            (
                F.col(
                    "Entertainment_Spend"
                )

                >

                F.col(
                    "Total_Spend"
                ) * 0.25
            ),

            "Entertainment Rewards Credit Card"
        )

        # ----------------------------------------------------
        # Clothing / shopping customer
        # ----------------------------------------------------

        .when(

            (
                F.col(
                    "Clothing_Spend"
                )

                >

                F.col(
                    "Total_Spend"
                ) * 0.25
            ),

            "Cashback Credit Card"
        )

        # ----------------------------------------------------
        # Senior customer
        # ----------------------------------------------------

        .when(

            F.col(
                "Age"
            ) >= 60,

            "Senior Citizen Fixed Deposit"
        )

        # ----------------------------------------------------
        # Default
        # ----------------------------------------------------

        .otherwise(
            "Premium Savings Account"
        )
    )
)


# ============================================================
# 20. RECOMMENDATION REASON
# ============================================================

product_affinity = (
    product_affinity

    .withColumn(

        "Recommendation_Reason",

        F.when(

            F.col(
                "Recommended_Product"
            ) == "Premium Dining Credit Card",

            "High restaurant spending"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Health Insurance",

            "High healthcare spending"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Wealth Management",

            "High account balance"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Business Loan",

            "Business account customer"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Entertainment Rewards Credit Card",

            "High entertainment spending"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Cashback Credit Card",

            "High shopping spending"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Senior Citizen Fixed Deposit",

            "Senior customer"
        )

        .otherwise(
            "General banking profile"
        )
    )
)


# ============================================================
# 21. RECOMMENDATION PRIORITY
# ============================================================

product_affinity = (
    product_affinity

    .withColumn(

        "Recommendation_Priority",

        F.when(

            F.col(
                "Customer_Value_Segment"
            ) == "HIGH_VALUE",

            "HIGH"
        )

        .when(

            F.col(
                "Customer_Value_Segment"
            ) == "MEDIUM_VALUE",

            "MEDIUM"
        )

        .otherwise(
            "LOW"
        )
    )
)


# ============================================================
# 22. NEXT BEST ACTION
# ============================================================

print("")
print("=" * 70)
print("CREATING NEXT BEST ACTION")
print("=" * 70)


next_best_action = (
    product_affinity

    .withColumn(

        "Next_Best_Action",

        F.when(

            F.col(
                "Recommended_Product"
            ) == "Premium Dining Credit Card",

            "Offer Premium Dining Credit Card"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Health Insurance",

            "Offer Health Insurance"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Wealth Management",

            "Schedule Wealth Management Consultation"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Business Loan",

            "Contact Customer for Business Loan"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Entertainment Rewards Credit Card",

            "Offer Entertainment Rewards Credit Card"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Cashback Credit Card",

            "Offer Cashback Credit Card"
        )

        .when(

            F.col(
                "Recommended_Product"
            ) == "Senior Citizen Fixed Deposit",

            "Offer Senior Citizen Fixed Deposit"
        )

        .otherwise(
            "Offer Premium Savings Account"
        )
    )
)


# ============================================================
# 23. PREFERRED COMMUNICATION CHANNEL
# ============================================================

next_best_action = (
    next_best_action

    .withColumn(

        "Preferred_Communication_Channel",

        F.when(

            F.col(
                "Preferred_Device"
            ).isin(

                "Mobile",
                "Mobile Device",
                "POS Mobile App"
            ),

            "MOBILE_APP"
        )

        .when(

            F.col(
                "Preferred_Device"
            ) == "Desktop",

            "EMAIL"
        )

        .when(

            F.col(
                "Preferred_Device"
            ) == "ATM",

            "SMS"
        )

        .otherwise(
            "EMAIL"
        )
    )
)


# ============================================================
# 24. CAMPAIGN TARGET LIST
# ============================================================

print("")
print("=" * 70)
print("CREATING CAMPAIGN TARGET LIST")
print("=" * 70)


campaign_target_list = (
    next_best_action

    .select(

        "Customer_ID",
        "Customer_Name",

        "Customer_Email",
        "Customer_Contact",

        "Account_Type",

        "Age",

        "State",
        "City",
        "Bank_Branch",

        "Total_Spend",

        "Average_Transaction_Amount",

        "Account_Balance",

        "Preferred_Merchant_Category",

        "Recommended_Product",

        "Recommendation_Reason",

        "Recommendation_Priority",

        "Next_Best_Action",

        "Preferred_Communication_Channel"
    )

    # --------------------------------------------------------
    # Email Eligibility
    # --------------------------------------------------------

    .withColumn(

        "Email_Eligible",

        F.when(

            F.col(
                "Customer_Email"
            ).isNotNull(),

            "YES"
        )

        .otherwise(
            "NO"
        )
    )

    # --------------------------------------------------------
    # SMS Eligibility
    # --------------------------------------------------------

    .withColumn(

        "SMS_Eligible",

        F.when(

            F.col(
                "Customer_Contact"
            ).isNotNull(),

            "YES"
        )

        .otherwise(
            "NO"
        )
    )

    # --------------------------------------------------------
    # Push Notification Eligibility
    # --------------------------------------------------------

    .withColumn(

        "Push_Notification_Eligible",

        F.when(

            F.col(
                "Preferred_Communication_Channel"
            ) == "MOBILE_APP",

            "YES"
        )

        .otherwise(
            "NO"
        )
    )

    # --------------------------------------------------------
    # Campaign Status
    # --------------------------------------------------------

    .withColumn(
        "Campaign_Status",
        F.lit("READY")
    )

    # --------------------------------------------------------
    # Created Timestamp
    # --------------------------------------------------------

    .withColumn(
        "Created_Timestamp",
        F.current_timestamp()
    )
)


# ============================================================
# 25. GOLD TABLE COLLECTION
# ============================================================

gold_tables = {

    "customer360":
        customer360,

    "spending_summary":
        spending_summary,

    "merchant_affinity":
        merchant_affinity,

    "digital_engagement":
        digital_engagement,

    "customer_segmentation":
        customer_segmentation,

    "customer_risk":
        customer_risk,

    "product_affinity":
        product_affinity,

    "next_best_action":
        next_best_action,

    "campaign_target_list":
        campaign_target_list
}


# ============================================================
# 26. WRITE GOLD TABLES TO S3 + DATABRICKS TABLES
# ============================================================

print("")
print("=" * 70)
print("WRITING GOLD TABLES")
print("=" * 70)

for table_name, dataframe in gold_tables.items():

    # --------------------------------------------------------
    # S3 path
    # --------------------------------------------------------

    table_path = (
        f"{GOLD_S3_PATH}/{table_name}"
    )

    # --------------------------------------------------------
    # Databricks table name
    # --------------------------------------------------------

    full_table_name = (
        f"{GOLD_SCHEMA}.{table_name}"
    )

    print("")
    print("-" * 70)
    print("Table:", table_name)
    print("S3 Path:", table_path)
    print("Databricks Table:", full_table_name)
    print("-" * 70)

    # --------------------------------------------------------
    # Write Delta data to S3
    # --------------------------------------------------------

    (
        dataframe

        .write

        .format("delta")

        .mode("overwrite")

        .option(
            "overwriteSchema",
            "true"
        )

        .save(
            table_path
        )
    )

    print(
        "S3 write completed:",
        table_path
    )

    # --------------------------------------------------------
    # Register external Delta table in Databricks
    # --------------------------------------------------------

    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS
        {full_table_name}
        USING DELTA
        LOCATION '{table_path}'
        """
    )

    print(
        "Databricks table registered:",
        full_table_name
    )


# ============================================================
# 27. GOLD TABLE VALIDATION
# ============================================================

print("")
print("=" * 70)
print("GOLD TABLE VALIDATION")
print("=" * 70)


for table_name in gold_tables.keys():

    full_table_name = (
        f"{GOLD_SCHEMA}.{table_name}"
    )

    table_path = (
        f"{GOLD_S3_PATH}/{table_name}"
    )

    record_count = (
        spark.table(
            full_table_name
        ).count()
    )

    print("")
    print(
        f"{full_table_name}"
    )

    print(
        f"Records      : {record_count}"
    )

    print(
        f"S3 Location  : {table_path}"
    )


# ============================================================
# 28. SHOW CUSTOMER 360
# ============================================================

print("")
print("=" * 70)
print("CUSTOMER 360")
print("=" * 70)


display(

    spark.table(
        f"{GOLD_SCHEMA}.customer360"
    )

    .limit(20)
)


# ============================================================
# 29. SHOW PRODUCT RECOMMENDATION
# ============================================================

print("")
print("=" * 70)
print("PRODUCT RECOMMENDATION")
print("=" * 70)


display(

    spark.table(
        f"{GOLD_SCHEMA}.product_affinity"
    )

    .select(

        "Customer_ID",

        "Customer_Name",

        "Account_Type",

        "Account_Balance",

        "Total_Spend",

        "Preferred_Merchant_Category",

        "Recommended_Product",

        "Recommendation_Reason",

        "Recommendation_Priority"
    )

    .limit(20)
)


# ============================================================
# 30. SHOW NEXT BEST ACTION
# ============================================================

print("")
print("=" * 70)
print("NEXT BEST ACTION")
print("=" * 70)


display(

    spark.table(
        f"{GOLD_SCHEMA}.next_best_action"
    )

    .select(

        "Customer_ID",

        "Customer_Name",

        "Recommended_Product",

        "Recommendation_Reason",

        "Next_Best_Action",

        "Recommendation_Priority",

        "Preferred_Communication_Channel"
    )

    .limit(20)
)


# ============================================================
# 31. SHOW CAMPAIGN TARGET LIST
# ============================================================

print("")
print("=" * 70)
print("CAMPAIGN TARGET LIST")
print("=" * 70)


display(

    spark.table(
        f"{GOLD_SCHEMA}.campaign_target_list"
    )

    .select(

        "Customer_ID",

        "Customer_Name",

        "Recommended_Product",

        "Next_Best_Action",

        "Preferred_Communication_Channel",

        "Email_Eligible",

        "SMS_Eligible",

        "Push_Notification_Eligible",

        "Campaign_Status"
    )

    .limit(20)
)


# ============================================================
# 32. SHOW GOLD S3 DIRECTORY
# ============================================================

print("")
print("=" * 70)
print("GOLD S3 OUTPUT")
print("=" * 70)

print("")
print("Root Gold Location:")
print(GOLD_S3_PATH)

print("")
print("Gold Tables:")

for table_name in gold_tables.keys():

    print(
        f" - {GOLD_S3_PATH}/{table_name}"
    )


# ============================================================
# 33. FINAL STATUS
# ============================================================

print("")
print("=" * 70)
print("SILVER -> GOLD COMPLETED SUCCESSFULLY")
print("=" * 70)

print("")
print("Silver Table:")
print(SILVER_TABLE)

print("")
print("Gold Schema:")
print(GOLD_SCHEMA)

print("")
print("Gold S3 Location:")
print(GOLD_S3_PATH)

print("")
print("Gold Tables:")

for table_name in gold_tables.keys():

    print(
        " -",
        f"{GOLD_SCHEMA}.{table_name}"
    )

print("")
print("S3 Locations:")

for table_name in gold_tables.keys():

    print(
        " -",
        f"{GOLD_S3_PATH}/{table_name}"
    )

print("")
print("Ready for:")

print("1. SageMaker ML recommendation model")
print("2. Amazon Bedrock personalized messaging")
print("3. Cross-sell / Up-sell")
print("4. Next Best Action")
print("5. Customer 360")
print("6. Campaign targeting")

print("")
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)